# Week 3 · pandas, method by method (hands-on)

Your log on Oct 25: *"it'd be better if i look into these [methods] and do some hands on applications or exercises."* So: one group of methods per section, each shown on a small factory table you can check by eye, then **an exercise on that group** with a checker. The checker tests your function on new random tables every run. Cheat sheet: McKinney, *Python for Data Analysis*, Ch. 5, 7, 8 and 10 (free online). Keep it open in another tab: looking a method up is normal.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **index**: The row labels of a DataFrame (here 0, 1, 2, ... unless you change them). `loc` uses labels; `iloc` uses positions.
- **mask**: A True/False Series, one value per row. `df[mask]` keeps the True rows, as in NumPy.
- **split-apply-combine**: What `groupby` does: split the rows into groups, apply a function to each group, then combine the results into one table.
- **returns a copy**: Almost every pandas method returns a **new** DataFrame and leaves the original alone (the `np.clip` rule from 00 §A3).

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from figures import *
import checks; checks.load(globals(), "03")      # defines check(n)

## 0. Recap first · 10 min, closed book

**Quick check:** kNN: `dist = [2.0, 0.5, 1.5, 3.0]`. What is `np.argsort(dist)[:2]`?

<details><summary>Answer</summary>

`[1 2]`: the **positions** of the two smallest values (0.5 at position 1, 1.5 at position 2). [02 §3 ↗](02_knn_and_numpy_workout.ipynb)

</details>

**Quick check:** `np.bincount([2, 0, 2, 1])` is ...? And its `argmax()`?

<details><summary>Answer</summary>

`[1 1 2]` (one 0, one 1, two 2s), and `argmax` is `2`. [02 §5 ↗](02_knn_and_numpy_workout.ipynb)

</details>

**Quick check:** `X` has shape `(100, 5)`. What are the shapes of `X.mean(axis=0)` and `X.mean()`?

<details><summary>Answer</summary>

`(5,)`, one per feature, and `()`, one number for everything. Scaling needs the first. [00 §A4 ↗](00_week2_review.ipynb)

</details>

**Quick check:** Why does logistic regression stop moving at the minimum?

<details><summary>Answer</summary>

The pulls cancel: $\sum_i (\hat p_i - y_i)\,x_i = 0$ and $\sum_i (\hat p_i - y_i) = 0$, so the gradient is zero. [01 §8 ↗](01_logistic_regression_in_pictures.ipynb)

</details>

**Quick check:** Recall vs precision: which one has the number of **alarms** in its denominator?

<details><summary>Answer</summary>

Precision: $\frac{TP}{TP + FP}$, out of everything flagged. Recall divides by the real positives, $TP + FN$. [Week 2 · 05 §4 ↗](../week2/05_logistic_regression_knn.ipynb)

</details>

## 1. Looking at a table

**The first five minutes with any dataset: how big is it, what type is each column, what values appear, and what's missing.**

Our table: 12 parts from a factory with three production lines. One temperature reading is missing (P04).

In [11]:
parts = pd.DataFrame({
    "part":     ["P01", "P02", "P03", "P04", "P05", "P06", "P07", "P08", "P09", "P10", "P11", "P12"],
    "line":     ["A", "A", "B", "C", "B", "A", "C", "B", "C", "A", "B", "C"],
    "shift":    ["day", "night", "day", "night", "night", "day", "day", "night", "day", "night", "day", "night"],
    "temp":     [71.2, 74.8, 69.5, np.nan, 77.1, 70.4, 72.9, 79.3, 68.8, 75.6, 70.1, 73.4],
    "pressure": [101, 98, 103, 99, 95, 102, 100, 94, 104, 97, 101, 99],
    "defect":   [0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0],
    "date":     ["2026-10-05", "2026-10-05", "2026-10-06", "2026-10-06", "2026-10-07", "2026-10-07",
                 "2026-10-08", "2026-10-08", "2026-10-09", "2026-10-09", "2026-10-10", "2026-10-10"],
})
print(parts.shape)            # → (12, 7)

(12, 7)


In [12]:
parts.head(3)        # the first rows, shown as a table (last line of a cell)

,part,line,shift,temp,pressure,defect,date
0,P01,A,day,71.2,101,0,2026-10-05
1,P02,A,night,74.8,98,1,2026-10-05
2,P03,B,day,69.5,103,0,2026-10-06


| Method | Answers | Gives back |
|---|---|---|
| `df.shape` | how many rows and columns? | a tuple `(12, 7)` |
| `df.head(n)` / `df.tail(n)` | what does it look like? | the first / last `n` rows |
| `df.dtypes` | what type is each column? | a Series: `float64`, `int64`, `str`, ... |
| `df.describe()` | range, mean and spread of each numeric column | a table of statistics |
| `s.value_counts()` | how often does each value appear? | a Series of counts, biggest first |
| `s.unique()` / `s.nunique()` | which values / how many distinct? | the values / a number |
| `df.isna().sum()` | how many missing per column? | a Series of counts |

In [13]:
print(parts["line"].value_counts().to_dict())       # → {'A': 4, 'B': 4, 'C': 4}
print(sorted(parts["line"].unique()), parts["line"].nunique())   # → ['A', 'B', 'C'] 3
print(parts.isna().sum().to_dict())
# → {'part': 0, 'line': 0, 'shift': 0, 'temp': 1, 'pressure': 0, 'defect': 0, 'date': 0}
print(parts[["temp", "pressure"]].describe().round(2))
# →         temp  pressure
#   count  11.00     12.00
#   mean   73.01     99.42
#   std     3.39      3.06
#   min    68.80     94.00
#   25%    70.25     97.75
#   50%    72.90     99.50
#   75%    75.20    101.25
#   max    79.30    104.00

{'A': 4, 'B': 4, 'C': 4}
['A', 'B', 'C'] 3
{'part': 0, 'line': 0, 'shift': 0, 'temp': 1, 'pressure': 0, 'defect': 0, 'date': 0}
        temp  pressure
count  11.00     12.00
mean   73.01     99.42
std     3.39      3.06
min    68.80     94.00
25%    70.25     97.75
50%    72.90     99.50
75%    75.20    101.25
max    79.30    104.00


`describe` counts 11 temperatures, not 12: missing values are skipped, not treated as zero. `.to_dict()` just prints a Series compactly.

✏️ **Exercise 1 · Summary.** Return a tuple `(n_rows, n_columns, n_missing)`, where `n_missing` is the total number of missing values in the whole table. Two methods from the table above.

In [16]:
def summary(df):
    n_rows, n_columns = df.shape
    n_missing = sum(df.isna().sum().to_dict().values())
    return(n_rows, n_columns, n_missing)

In [17]:
check(1)

Exercise 1: PASS


## 2. Picking rows and columns

**Columns by name with `[]`; rows and columns by label with `loc`; by position with `iloc`; rows by condition with a mask.**

🖼 **Picture.** The same 6 rows of `parts` in every panel. **Yellow = what that line of code gives back.** Grey numbers on the left are the row labels; grey `col 0`–`col 3` are the column positions.

In [ ]:
fig_select()

| You write | You pick by | Example | Watch out |
|---|---|---|---|
| `df["name"]` / `df[["a", "b"]]` | column name | `parts["temp"]` | one name → Series; a list → DataFrame |
| `df.loc[rows, cols]` | row **label**, column **name** | `parts.loc[0:2, ["part", "temp"]]` | `0:2` **includes** label 2 |
| `df.iloc[rows, cols]` | row and column **position** | `parts.iloc[0:2, 1:3]` | `0:2` **stops before** 2, like NumPy |
| `df[mask]` | a True/False per row | `parts[parts["temp"] > 74]` | keeps whole rows |

In [21]:
print(parts.loc[2, "temp"])                    # → 69.5
print(parts.loc[0:2, ["part", "temp"]])
# →   part  temp
#   0  P01  71.2
#   1  P02  74.8
#   2  P03  69.5
print(parts.iloc[0:2, 1:3])
# →   line  shift
#   0    A    day
#   1    A  night

69.5
  part  temp
0  P01  71.2
1  P02  74.8
2  P03  69.5
  line  shift
0    A    day
1    A  night


**Masks with several conditions.** Combine them with `&` (and), `|` (or) and `~` (not). **Every condition needs its own parentheses**, because `&` binds tighter than `==` and `>`.

In [22]:
night_hot = parts[(parts["shift"] == "night") & (parts["temp"] > 74)]
print(night_hot[["part", "shift", "temp"]])
# →   part  shift  temp
#   1  P02  night  74.8
#   4  P05  night  77.1
#   7  P08  night  79.3
#   9  P10  night  75.6
print(parts[parts["line"].isin(["A", "C"])]["part"].tolist())
# → ['P01', 'P02', 'P04', 'P06', 'P07', 'P09', 'P10', 'P12']
print(parts.query("pressure < 98 and defect == 1")["part"].tolist())   # → ['P05', 'P08', 'P10']

  part  shift  temp
1  P02  night  74.8
4  P05  night  77.1
7  P08  night  79.3
9  P10  night  75.6
['P01', 'P02', 'P04', 'P06', 'P07', 'P09', 'P10', 'P12']
['P05', 'P08', 'P10']


> ⚠️ **Common mistake.** `parts[parts["shift"] == "night" & parts["temp"] > 74]` (no parentheses) raises an error, because Python reads it as `"night" & parts["temp"]` first. And `and`/`or` don't work on Series at all: use `&`/`|`.

✏️ **Exercise 2 · Hot night parts.** Return the rows where `shift` is `"night"` **and** `temp` is greater than `t`, as a DataFrame (all columns, original index).

In [25]:
def hot_night(df, t):
    return df[(df["shift"] == "night") & (df["temp"] > t)]

In [26]:
check(2)

Exercise 2: PASS


✏️ **Exercise 3 · Parts on some lines.** Return a **list** of the `part` ids whose `line` is in the list `lines`, in table order. Use `isin`, then `.tolist()`.

In [29]:
def parts_on_lines(df, lines):
    return df[(df["line"].isin(lines))]["part"].tolist()

In [30]:
check(3)

Exercise 3: PASS


## 3. Sorting

**`sort_values` reorders rows by one or more columns. `nlargest`/`nsmallest` are shortcuts for the top $k$. `reset_index(drop=True)` renumbers the rows 0, 1, 2, ...**

In [ ]:
print(parts.sort_values("temp", ascending=False).head(3)[["part", "temp"]])
# →   part  temp
#   7  P08  79.3
#   4  P05  77.1
#   9  P10  75.6
print(parts.sort_values(["line", "temp"])[["part", "line", "temp"]].head(4))   # by line, then temp inside each line
# →   part line  temp
#   5  P06    A  70.4
#   0  P01    A  71.2
#   1  P02    A  74.8
#   9  P10    A  75.6
print(parts.nlargest(3, "temp")[["part", "temp"]].reset_index(drop=True))
# →   part  temp
#   0  P08  79.3
#   1  P05  77.1
#   2  P10  75.6

After sorting, the index keeps the **old** labels (7, 4, 9). That's useful for tracing rows back, but confusing if you then use `loc[0]`. `reset_index(drop=True)` throws the old labels away.

✏️ **Exercise 4 · Top k.** Return the `k` rows with the largest values in column `col`, biggest first, with the index reset to `0, 1, ..., k - 1`.

In [33]:
def top_k(df, col, k):
    return df.nlargest(k, col).reset_index(drop=True)

In [34]:
check(4)

Exercise 4: PASS


## 4. New columns

**Arithmetic on whole columns makes new columns (no loops, as in NumPy). `np.where` picks between two values per row, and `map` translates values through a dictionary.**

In [35]:
p2 = parts.assign(temp_f=parts["temp"] * 9 / 5 + 32,              # a new column from a formula
                  hot=np.where(parts["temp"] > 75, "yes", "no"))   # pick per row
print(p2[["part", "temp", "temp_f", "hot"]].head(4))
# →   part  temp  temp_f hot
#   0  P01  71.2  160.16  no
#   1  P02  74.8  166.64  no
#   2  P03  69.5  157.10  no
#   3  P04   NaN     NaN  no
print(parts["shift"].map({"day": 0, "night": 1}).tolist())   # → [0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1]
print(parts.shape)            # → (12, 7)   assign returned a NEW table; parts is unchanged

  part  temp  temp_f hot
0  P01  71.2  160.16  no
1  P02  74.8  166.64  no
2  P03  69.5  157.10  no
3  P04   NaN     NaN  no
[0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1]
(12, 7)


| Method | Use it for |
|---|---|
| `df["new"] = ...` | add a column **in place** (changes `df`) |
| `df.assign(new=...)` | the same, but returns a new table (the original stays as it was) |
| `np.where(cond, a, b)` | two-way choice per row |
| `s.map(dict)` | translate values: codes → names, or names → numbers |
| `s.astype(int)` / `float` / `str` | change the type |
| `df.rename(columns={"old": "new"})` | rename columns |
| `df.drop(columns=[...])` | remove columns |
| `s.apply(f)` | any Python function, one row at a time: **slow**, so use it only when there's no vectorized way |

P04's temperature is missing, so its `temp_f` is missing too (NaN in, NaN out), and `NaN > 75` is False, which gives "no".

✏️ **Exercise 5 · Two new columns.** Return a **new** DataFrame (don't change `df`) with two extra columns: `temp_f` (the temperature in Fahrenheit, $\frac{9}{5}t + 32$) and `hot` (`True`/`False`: `temp > 75`). The checker verifies that the original `df` is unchanged.

In [46]:
def add_columns(df):
    return df.assign(temp_f = df["temp"] * 9/5 + 32, hot = np.where(df["temp"] > 75, True, False))

In [47]:
check(5)

Exercise 5: PASS


## 5. Missing values

**Find them with `isna`, then either drop the rows (`dropna`) or fill them in (`fillna`). A smarter fill uses the mean of similar rows.**

### `dropna(subset=[...])`

Removes rows with missing values (`NaN`), but only checks the columns you list.

- `df.dropna()`: drop a row if **any** column is missing.
- `df.dropna(subset=["temp"])`: drop a row only if `temp` is missing.
- `subset=["temp", "qty"]`: drop if **either** is missing.
- `.shape` afterwards gives `(rows, columns)`, so `parts.dropna(subset=["temp"]).shape[0]` counts the rows that have a temperature.
- It returns a new DataFrame. To keep the result: `parts = parts.dropna(subset=["temp"])`.

### `groupby` and `.transform()`

**`parts.groupby("line")`** returns a `DataFrameGroupBy` object: a grouping plan, nothing computed yet. Inspect it with `.groups` (row indices per group), `.size()`, `.get_group("A")`, or `for name, rows in g:`. Adding `["temp"]` narrows it to one column (`SeriesGroupBy`).

**`.transform("mean")`**: split → apply → broadcast back.
1. Split `temp` by line: A = `[20, 24, 22]`, B = `[30, 34]`
2. Apply the function per group: A → 22, B → 32
3. Copy each group's result onto every row of that group
4. Return in the original row order: **same length and index as the input**

| | Result | Length |
|---|---|---|
| `.groupby(...)["temp"].mean()` | one value per group | number of groups |
| `.groupby(...)["temp"].transform("mean")` | group value on each row | same as the table |

Because it lines up row by row, it combines directly with the original columns:
```python
line_mean = parts.groupby("line")["temp"].transform("mean")
parts["temp"] - line_mean           # distance from own line's average
parts["temp"].fillna(line_mean)     # fill missing temps with own line's average
```

Other functions: `"sum"`, `"count"`, `"max"`, or a custom one like `lambda s: s - s.mean()` (center within each group).

In [48]:
print(parts.dropna(subset=["temp"]).shape)                       # → (11, 7)   P04 dropped
print(parts["temp"].fillna(parts["temp"].mean()).round(2).tolist()[:5])   # → [71.2, 74.8, 69.5, 73.01, 77.1]
line_mean = parts.groupby("line")["temp"].transform("mean")       # each row gets ITS line's mean
print(line_mean.round(2).tolist()[:5])                            # → [73.0, 73.0, 74.0, 71.7, 74.0]
print(parts["temp"].fillna(line_mean).round(2).tolist()[:5])      # → [71.2, 74.8, 69.5, 71.7, 77.1]

(11, 7)
[71.2, 74.8, 69.5, 73.01, 77.1]
[73.0, 73.0, 74.0, 71.7, 74.0]
[71.2, 74.8, 69.5, 71.7, 77.1]


P04 is on line C. Filling with the overall mean gives 73.01; filling with **line C's** mean (68.8, 72.9, 73.4 → 71.7) is closer to what its line actually runs at. `transform` is explained in the next section.

> ⚠️ **Common mistake.** **Fill with statistics from the training set only.** Filling the test set with its own mean is the same leakage as fitting a scaler on all the data ([Week 2 · 05 §4 ↗](../week2/05_logistic_regression_knn.ipynb)). Inside a scikit-learn pipeline, `SimpleImputer` does it right.

## 6. groupby: split, apply, combine

**`df.groupby("line")["temp"].mean()` splits the rows by line, takes the mean temperature inside each group, and puts the results together, one row per group.**

🖼 **Picture.** `parts.groupby("line")["temp"].mean()`, drawn. Rows are coloured by their line: blue = A, orange = B, grey = C.

In [ ]:
fig_groupby()

- **1. Split.** pandas sorts the 12 rows into one pile per value of `line`: 4 rows for A, 4 for B, 4 for C.
- **2. Apply.** Inside each pile it takes the mean of `temp`. Line A: $\frac{71.2 + 74.8 + 70.4 + 75.6}{4} = 73.0$. Line C has a missing value (P04), which is skipped, so its mean uses 3 values: $\frac{72.9 + 68.8 + 73.4}{3} = 71.7$.
- **3. Combine.** The three answers become one small table with one row per line. The line names become the row labels (the index).

In [49]:
print(parts.groupby("line")["defect"].mean().to_dict())    # → {'A': 0.5, 'B': 0.5, 'C': 0.0}
print(parts.groupby("shift")["defect"].agg(["sum", "mean", "count"]).round(3))
# →        sum   mean  count
#   shift
#   day      0  0.000      6
#   night    4  0.667      6
stats = parts.groupby("line").agg(n=("part", "size"), mean_temp=("temp", "mean"), defects=("defect", "sum"))
print(stats.round(2))
# →       n  mean_temp  defects
#   line
#   A     4       73.0        2
#   B     4       74.0        2
#   C     4       71.7        0

{'A': 0.5, 'B': 0.5, 'C': 0.0}
       sum   mean  count
shift                   
day      0  0.000      6
night    4  0.667      6
      n  mean_temp  defects
line                       
A     4       73.0        2
B     4       74.0        2
C     4       71.7        0


Every defect happened on the **night** shift. That's a pattern worth a question to the factory, and the kind of thing you find in five minutes with `groupby`.

| After `groupby(...)` | Gives | Shape |
|---|---|---|
| `.mean()`, `.sum()`, `.max()`, ... | one value per group | one row per group |
| `.agg(name=(col, func), ...)` | several named summaries | one row per group |
| `.size()` | the number of rows per group (missing values included) | one row per group |
| `.transform("mean")` | the group's value written back on **every row** | same as the original table |

🖼 **Picture.** `.mean()` vs `.transform("mean")`: the same three numbers, but `transform` writes each one back onto every row of its group, so the result lines up with `parts` row by row. That's what exercise 6 needs.

In [ ]:
fig_transform()

✏️ **Exercise 6 · Fill by line.** Return a copy of `df` where each missing `temp` is replaced by the mean temperature of **its own line**. Use `groupby("line")["temp"].transform("mean")` and `fillna`. Don't change `df`.

In [79]:
def fill_temp_by_line(df):
    out = df.copy()
    line_mean = out.groupby("line")["temp"].transform("mean")
    out["temp"] = out["temp"].fillna(line_mean)
    return out

In [80]:
check(6)

Exercise 6: PASS


✏️ **Exercise 7 · Defect rate by group.** Return the mean of `defect` for each value of column `col` (e.g. `"line"` or `"shift"`), as a Series indexed by those values.

In [81]:
def defect_rate_by(df, col):
    return df.groupby(col)["defect"].mean()

In [82]:
check(7)

Exercise 7: PASS


✏️ **Exercise 8 · Line statistics.** Return a DataFrame with one row per line and exactly these columns, in this order: `n` (number of parts), `mean_temp`, `max_pressure`. Use named aggregation, as in `stats` above.

In [90]:
def line_stats(df):
    stats = df.groupby("line").agg(n=("part", "size"), mean_temp=("temp", "mean"), max_pressure=("pressure", "max"))
    return stats

In [91]:
check(8)

Exercise 8: PASS


## 7. Combining tables: merge and concat

**`merge` puts two tables side by side by matching the values of a shared column, the key. Each row of the left table looks for the row of the right table with the same key and is glued to it.** It's SQL's `JOIN`.

Two small tables first, so every row is visible: four parts with their line, and a table saying who supervises each line. Note that there's no supervisor for line C, and line D has no parts.

In [ ]:
small = parts[["part", "line"]].head(4)
lines = pd.DataFrame({"line": ["A", "B", "D"], "supervisor": ["Kim", "Lee", "Park"]})
print(small)
# →   part line
#   0  P01    A
#   1  P02    A
#   2  P03    B
#   3  P04    C
print(lines)
# →   line supervisor
#   0    A        Kim
#   1    B        Lee
#   2    D       Park

🖼 **Picture.** Top: each row of `small` looks up its line in `lines` (green arrows). Bottom: what the three most common `how=` options keep. Colours follow the line: blue = A, orange = B, grey = C, purple = D; red = NaN (nothing found).

In [ ]:
fig_merge()

**How merge builds the result, row by row**

- P01 has line A. Look up A in `lines`: found, Kim. Result row: P01, A, Kim.
- P02 has line A: Kim again. **One right row can match many left rows.**
- P03 has line B: Lee.
- P04 has line C: there is **no** C in `lines`. Now `how` decides: `inner` drops P04, while `left` keeps it with supervisor NaN.
- Park (line D) is never looked up, because no part is on line D. Only `how="outer"` (or `"right"`) keeps him, with part NaN.

| `how=` | Keeps | Rows here | SQL |
|---|---|---|---|
| `"inner"` (default) | only rows whose key is in **both** tables | 3: P01, P02, P03 | `INNER JOIN` |
| `"left"` | **every** row of the left table; NaN where no partner | 4: + P04 with NaN | `LEFT JOIN` |
| `"right"` | every row of the right table | 4: P01, P02, P03 + Park with NaN | `RIGHT JOIN` |
| `"outer"` | every row of both tables | 5 | `FULL OUTER JOIN` |

In [ ]:
print(small.merge(lines, on="line", how="inner"))
# →   part line supervisor
#   0  P01    A        Kim
#   1  P02    A        Kim
#   2  P03    B        Lee
print(small.merge(lines, on="line", how="left"))
# →   part line supervisor
#   0  P01    A        Kim
#   1  P02    A        Kim
#   2  P03    B        Lee
#   3  P04    C        NaN

**On the full table.** `parts` has 4 parts on line C, so `how="inner"` loses all four, while `how="left"` keeps all 12 rows:

In [ ]:
print(parts.merge(lines, on="line", how="inner").shape)   # → (8, 8)    12 parts − 4 on line C
print(parts.merge(lines, on="line", how="left").shape)    # → (12, 8)   every part kept

**`concat` matches nothing: it just stacks tables**, the rows of the first, then the rows of the second (with `axis=1`, side by side instead). Use it when the tables have the same columns, e.g. this week's data plus last week's:

In [ ]:
print(pd.concat([parts.head(2), parts.tail(1)])["part"].tolist())   # → ['P01', 'P02', 'P12']

✏️ **Exercise 9 · Attach supervisors.** `sup` is a table with columns `line` and `supervisor`. Return `df` with a `supervisor` column added, keeping **every** part (NaN where its line isn't in `sup`).

In [92]:
def attach_supervisor(df, sup):
    return df.merge(sup, on = "line", how ="left")

In [93]:
check(9)

Exercise 9: PASS


## 8. Reshaping: pivot tables and crosstabs

**`pivot_table` turns one column's values into column headers, so that two groupings can be read as a grid.**

🖼 **Picture.** Left: the number of defects for every (line, shift) pair, as a long list. Right: the same six numbers as a grid. Each long row becomes one cell.

In [ ]:
fig_pivot()

In [ ]:
print(parts.pivot_table(index="line", columns="shift", values="defect", aggfunc="sum", fill_value=0))
# → shift  day  night
#   line
#   A        0      2
#   B        0      2
#   C        0      0
print(pd.crosstab(parts["line"], parts["defect"]))     # counts of each (line, defect) pair
# → defect  0  1
#   line
#   A       2  2
#   B       2  2
#   C       4  0

Read the pivot table as "defects by line (rows) and shift (columns)". It's `groupby(["line", "shift"])["defect"].sum()` laid out as a grid instead of a long list.

✏️ **Exercise 10 · Defects grid.** Return the pivot table of the **number of defects** (sum of `defect`) with lines as rows and shifts as columns, with 0 where a combination never occurs.

In [ ]:
def defects_table(df):
    pass  # your code

In [ ]:
check(10)

## 9. Dates and text

**`pd.to_datetime` turns date strings into real dates, and then `.dt` gives the parts of each date. `.str` does the same for text columns.**

In [ ]:
d = pd.to_datetime(parts["date"])
print(d.dt.day_name().head(3).tolist())       # → ['Monday', 'Monday', 'Tuesday']
print(d.dt.day.head(3).tolist())              # → [5, 5, 6]   the day of the month
print(parts["part"].str.replace("P", "part-").head(2).tolist())   # → ['part-01', 'part-02']
print(parts["shift"].str.upper().head(2).tolist())                # → ['DAY', 'NIGHT']

Time series (your KAMP data) live on `.dt`: `.dt.hour`, `.dt.dayofweek` (Monday = 0), `.dt.month`. They make good features.

✏️ **Exercise 11 · Parts per weekday.** Return how many parts were made on each **weekday name** (`"Friday"`, `"Monday"`, ...), as a Series sorted alphabetically by weekday name. Hint: `to_datetime`, `.dt.day_name()`, `value_counts()`, `sort_index()`.

In [ ]:
def weekday_counts(df):
    pass  # your code

In [ ]:
check(11)

## 10. From pandas to scikit-learn

**Models need numbers. `pd.get_dummies` turns each text category into its own 0/1 column (one-hot encoding), and `.to_numpy()` hands the result to scikit-learn.**

In [ ]:
X = pd.get_dummies(parts[["temp", "pressure", "line", "shift"]].dropna(), columns=["line", "shift"], dtype=float)
print(X.columns.tolist())
# → ['temp', 'pressure', 'line_A', 'line_B', 'line_C', 'shift_day', 'shift_night']
print(X.head(2))
# →    temp  pressure  line_A  line_B  line_C  shift_day  shift_night
#   0  71.2       101     1.0     0.0     0.0        1.0          0.0
#   1  74.8        98     1.0     0.0     0.0        0.0          1.0
print(X.to_numpy().shape)     # → (11, 7)

Why not just code the lines as A = 0, B = 1, C = 2? Because then a model would think C is "twice" B and that B lies between A and C. One-hot columns say only "is it this line or not". It's your `one_hot` from [02, exercise 15 ↗](02_knn_and_numpy_workout.ipynb), done by pandas.

✏️ **Exercise 12 · Features and labels.** Return `(X, y)` as NumPy arrays: take the columns `temp`, `pressure`, `line`, `shift`, `defect`; drop rows with any missing value; one-hot encode `line` and `shift` with `pd.get_dummies(..., columns=["line", "shift"], dtype=float)`; then `y` is the `defect` column and `X` is everything else, in the order `get_dummies` gives.

In [ ]:
def to_xy(df):
    pass  # your code

In [ ]:
check(12)

In [ ]:
for n in range(1, 13):
    check(n)

---
Solutions: `solutions/03_solutions.py`. Next: `04_decision_trees.ipynb`, where the ML syllabus continues.